In [1]:
# =========================================
# 3.5 Reformatting (pre-Transformation base tables)
# Inputs:
#   prep_outputs/final_integrated_table_semantic.csv
# Outputs:
#   prep_outputs/final_reporting_table.csv / .parquet
#   prep_outputs/final_modeling_table_reformatted.csv / .parquet
# Notes:
#   - This is NOT the final modeling table; it's the base for Step 4 (Transformation).
#   - Keep *_LABEL only in the reporting table; drop them for the modeling base.
# =========================================

import os
import numpy as np
import pandas as pd
from pathlib import Path

IN_CSV = Path("prep_outputs/final_integrated_table_semantic.csv")
OUT_DIR = Path("prep_outputs")
OUT_DIR.mkdir(parents=True, exist_ok=True)

assert IN_CSV.exists(), f"Input not found: {IN_CSV}"
df_sem = pd.read_csv(IN_CSV)

# -----------------------------
# 1) Column partitions
# -----------------------------
label_cols = [c for c in df_sem.columns if c.endswith("_LABEL")]
core_cols  = [c for c in df_sem.columns if c not in label_cols]

# Keep a conventional order: RID, AMIGR, then rest
RID_COL = "RID"
TARGET_COL = "AMIGR"

# Ensure RID & Target are first if present
def ordered(cols):
    pref = [x for x in [RID_COL, TARGET_COL] if x in cols]
    rest = [c for c in cols if c not in pref]
    return pref + rest

# Reporting: RID, AMIGR, *_LABEL only
reporting_cols = [RID_COL, TARGET_COL] + label_cols
reporting_cols = [c for c in reporting_cols if c in df_sem.columns]
df_reporting = df_sem[ordered(reporting_cols)].copy()

# Modeling base: drop *_LABEL
df_modeling_base = df_sem[ordered(core_cols)].copy()

# -----------------------------
# 2) DType harmonization (light-touch)
#    - Keep numeric predictors numeric
#    - Enforce RID/TARGET integer if possible
#    - Keep LABEL columns as string
# -----------------------------
# LABELs as string (reporting only)
for c in label_cols:
    if c in df_reporting.columns:
        df_reporting[c] = df_reporting[c].astype("string")

# Try to enforce RID/TARGET as integer in both tables (nullable-safe)
for frame in (df_reporting, df_modeling_base):
    if RID_COL in frame.columns:
        frame[RID_COL] = pd.to_numeric(frame[RID_COL], errors="coerce").astype("Int64")
    if TARGET_COL in frame.columns:
        frame[TARGET_COL] = pd.to_numeric(frame[TARGET_COL], errors="coerce").astype("Int64")

# Optional: integer-count variables (re-validate & clamp; already handled before, but safe)
int_count_ranges = {
    "ALC12MYR": (0, 365),
    "BEDDAYR":  (0, 365),
    "YRSWRKPA": (0, 35),
}
for frame in (df_modeling_base,):
    for col, (lo, hi) in int_count_ranges.items():
        if col in frame.columns:
            s = pd.to_numeric(frame[col], errors="coerce")
            s = np.rint(s).astype("Int64")
            s = s.clip(lower=lo, upper=hi)
            frame[col] = s

# -----------------------------
# 3) Minimal QA checks
# -----------------------------
def qa_print(frame, name):
    n_rows, n_cols = frame.shape
    n_missing_rows = frame.isna().any(axis=1).sum()
    print(f"[QA] {name}: shape={frame.shape} | rows-with-NA={n_missing_rows}/{n_rows}")

qa_print(df_reporting, "reporting")
qa_print(df_modeling_base, "modeling_base")

# Check duplicates on RID (if RID exists)
for frame, name in ((df_reporting, "reporting"), (df_modeling_base, "modeling_base")):
    if RID_COL in frame.columns:
        dup = frame[RID_COL].duplicated(keep=False).sum()
        print(f"[QA] {name}: duplicated RID count = {dup}")

# Row alignment sanity (if both have RID)
if RID_COL in df_reporting.columns and RID_COL in df_modeling_base.columns:
    aligned = df_reporting[RID_COL].reset_index(drop=True).fillna(-1).astype("Int64").equals(
        df_modeling_base[RID_COL].reset_index(drop=True).fillna(-1).astype("Int64")
    )
    print(f"[QA] RID alignment between reporting and modeling_base: {aligned}")



# -----------------------------
# 4) Save outputs (CSV + Parquet)
# -----------------------------
REPORT_CSV = OUT_DIR / "final_reporting_table.csv"
MODEL_CSV  = OUT_DIR / "final_modeling_table_reformatted.csv"
REPORT_PQ  = OUT_DIR / "final_reporting_table.parquet"
MODEL_PQ   = OUT_DIR / "final_modeling_table_reformatted.parquet"

df_reporting.to_csv(REPORT_CSV, index=False)
df_modeling_base.to_csv(MODEL_CSV, index=False)

# Parquet can be more efficient for downstream transformation
try:
    df_reporting.to_parquet(REPORT_PQ, index=False)
    df_modeling_base.to_parquet(MODEL_PQ, index=False)
    print(f"[OK] Saved CSV and Parquet:")
    print(f"     - {REPORT_CSV.name} / {REPORT_PQ.name}")
    print(f"     - {MODEL_CSV.name}  / {MODEL_PQ.name}")
except Exception as e:
    print("[Warn] Parquet save failed (missing pyarrow/fastparquet?). CSVs were saved.")
    print("       Error:", e)

print("[Done] 3.5 Reformatting completed (base tables for Step 4).")


[QA] reporting: shape=(25403, 47) | rows-with-NA=0/25403
[QA] modeling_base: shape=(25403, 58) | rows-with-NA=0/25403
[QA] reporting: duplicated RID count = 0
[QA] modeling_base: duplicated RID count = 0
[QA] RID alignment between reporting and modeling_base: True
[Warn] Parquet save failed (missing pyarrow/fastparquet?). CSVs were saved.
       Error: Unable to find a usable engine; tried using: 'pyarrow', 'fastparquet'.
A suitable version of pyarrow or fastparquet is required for parquet support.
Trying to import the above resulted in these errors:
 - Missing optional dependency 'pyarrow'. pyarrow is required for parquet support. Use pip or conda to install pyarrow.
 - Missing optional dependency 'fastparquet'. fastparquet is required for parquet support. Use pip or conda to install fastparquet.
[Done] 3.5 Reformatting completed (base tables for Step 4).
